# U.S. Avocado Market Trends, 2026 YTD vs. 2025

Reproduces every figure cited in the memo to April Morris (September 24, 2026).

**Inputs**
- `HAB-FlatFile-Weekly-2025.csv`: full-year 2025, weeks 1–52
- `HAB-FlatFile-Weekly-2026-P1-P8.csv`: 2026, weeks 1–32, through August 9

**How to run:** keep the two CSV files in the same folder as this notebook (or in Downloads, Desktop, Documents or OneDrive), then choose *Kernel → Restart & Run All*.

**Method notes**
- National figures use the "Total U.S." rows (never a sum of markets, which would double-count regions and metros).
- Year-over-year comparisons use the same weeks (1–32) in both years.
- Average selling price is volume-weighted: sum(price × units) / sum(units).
- Retail dollars are **estimated** as price × units; the files have no dollar column.

## Setup: locate the data files
The notebook finds the two HAB CSV files automatically and prints the paths it used. If it can't find them, set `F2025` and `F2026` to the full file paths.

In [13]:
import pandas as pd
from pathlib import Path

# --- File locations -------------------------------------------------------
# The notebook searches for the two HAB CSV files automatically (the notebook's
# own folder first, then Downloads, Desktop, Documents and OneDrive).
# To point at specific files instead, replace None with the full path, e.g.
#   F2025 = r"C:\Users\alfredo\Downloads\HAB-FlatFile-Weekly-2025.csv"
F2025 = None
F2026 = None


def find_csv(pattern):
    """Return the newest file matching `pattern` in the usual folders."""
    home = Path.home()
    folders = [Path.cwd(), home / "Downloads", home / "Desktop", home / "Documents",
               home / "OneDrive", home / "OneDrive" / "Desktop",
               home / "OneDrive" / "Documents"]
    for folder in folders:
        if not folder.exists():
            continue
        hits = list(folder.glob(pattern)) or list(folder.rglob(pattern))
        if hits:
            return max(hits, key=lambda p: p.stat().st_mtime)
    raise FileNotFoundError(
        f"Could not find a file matching {pattern!r}. "
        "Set F2025 / F2026 above to the full file paths.")


F2025 = Path(F2025) if F2025 else find_csv("HAB-FlatFile-Weekly-2025*.csv")
F2026 = Path(F2026) if F2026 else find_csv("HAB-FlatFile-Weekly-2026*.csv")
print("2025 file:", F2025)
print("2026 file:", F2026)


YTD_WEEKS = 32
REGIONS = ["California", "Great Lakes", "Midsouth", "Northeast",
           "Plains", "South Central", "Southeast", "West"]
# Metro markets assigned to the Southeast region (HAB regional grouping)
SOUTHEAST_MARKETS = ["Atlanta", "Birmingham/Montgomery", "Charlotte", "Jacksonville",
                     "Miami/FT. Lauderdale", "Orlando", "Raleigh/Greensboro",
                     "Richmond/Norfolk", "Roanoke", "Tampa"]

VOL = "Total Bulk and Bags"
ASP = "ASP Current Year"

2025 file: C:\Users\alfre\Downloads\HAB-FlatFile-Weekly-2025 (1).csv
2026 file: C:\Users\alfre\Downloads\HAB-FlatFile-Weekly-2026-P1-P8 (1).csv


## Helper functions

In [14]:
def load():
    df = pd.concat([pd.read_csv(F2025), pd.read_csv(F2026)], ignore_index=True)
    df["date"] = pd.to_datetime(df["Current Year Week Ending"])
    df["year"] = df["date"].dt.year
    df["dollars"] = df[ASP] * df[VOL]          # estimated retail dollars
    return df


def summarize(frame, by):
    """Units, estimated dollars and volume-weighted ASP grouped by `by`."""
    g = frame.groupby(by)[[VOL, "dollars"]].sum()
    g["asp"] = g["dollars"] / g[VOL]
    return g


def pct(new, old):
    return (new / old - 1) * 100


def header(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## Load data

In [15]:
df = load()
us = df[df["Geography"] == "Total U.S."]
us_ytd = us[us["Period"] <= YTD_WEEKS]
conv = us[us["Type"] == "Conventional"]
org = us[us["Type"] == "Organic"]

## 1. Summary table

In [16]:
header("1. SUMMARY TABLE - Total U.S., weeks 1-32")
tot = summarize(us_ytd, "year")
print(f"Units sold (millions):     {tot.loc[2025, VOL]/1e6:,.0f} -> "
      f"{tot.loc[2026, VOL]/1e6:,.0f}  ({pct(tot.loc[2026, VOL], tot.loc[2025, VOL]):+.1f}%)")
print(f"Average selling price:     ${tot.loc[2025, 'asp']:.2f} -> "
      f"${tot.loc[2026, 'asp']:.2f}  ({pct(tot.loc[2026, 'asp'], tot.loc[2025, 'asp']):+.1f}%)")
print(f"Retail dollars (millions): ${tot.loc[2025, 'dollars']/1e6:,.0f} -> "
      f"${tot.loc[2026, 'dollars']/1e6:,.0f}  "
      f"({pct(tot.loc[2026, 'dollars'], tot.loc[2025, 'dollars']):+.1f}%)")


1. SUMMARY TABLE - Total U.S., weeks 1-32
Units sold (millions):     1,871 -> 2,126  (+13.6%)
Average selling price:     $1.25 -> $1.01  (-19.1%)
Retail dollars (millions): $2,329 -> $2,143  (-8.0%)


## 2. Prices moved in two phases

In [17]:
header("2. PRICE PHASES - conventional, Total U.S.")
jan_may = summarize(conv[conv["date"].dt.month <= 5], "year")
print(f"Jan-May avg price: 2025 ${jan_may.loc[2025, 'asp']:.3f}, "
      f"2026 ${jan_may.loc[2026, 'asp']:.3f}  "
      f"({pct(jan_may.loc[2026, 'asp'], jan_may.loc[2025, 'asp']):+.1f}%)")

wk = conv.set_index(["year", "Period"])[[ASP, VOL]].sort_index()
w26 = wk.loc[2026]
peak_week = w26[ASP].idxmax()
vol_yoy_peak = pct(w26.loc[peak_week, VOL], wk.loc[(2025, peak_week), VOL])
print(f"2026 peak price: ${w26[ASP].max():.2f} in week {peak_week}; "
      f"volume vs. same week 2025: {vol_yoy_peak:+.1f}% (near zero)")
print(f"Latest week ({w26.index.max()}): ${w26[ASP].iloc[-1]:.2f}")

print("\n2026 weekly conventional price and volume (millions):")
print(pd.DataFrame({"price": w26[ASP].round(2),
                    "units_M": (w26[VOL] / 1e6).round(1)}).to_string())

late25 = wk.loc[2025].loc[33:52, ASP]
first_below_1 = late25[late25 < 1].index.min()
print(f"\nLate-2025 trend: ${late25.iloc[0]:.2f} (wk 33) -> ${late25.iloc[-1]:.2f} (wk 52); "
      f"first week below $1.00: week {first_below_1}")


2. PRICE PHASES - conventional, Total U.S.
Jan-May avg price: 2025 $1.227, 2026 $0.930  (-24.2%)
2026 peak price: $1.19 in week 26; volume vs. same week 2025: +1.5% (near zero)
Latest week (32): $1.06

2026 weekly conventional price and volume (millions):
        price  units_M
Period                
1        0.92     60.9
2        0.92     69.6
3        0.93     65.4
4        0.93     67.6
5        0.92     59.1
6        0.89     76.0
7        0.91     61.4
8        0.90     65.3
9        0.92     63.1
10       0.92     66.4
11       0.93     64.9
12       0.92     64.9
13       0.91     66.2
14       0.95     63.8
15       0.94     66.7
16       0.94     67.6
17       0.94     66.8
18       0.92     71.8
19       0.94     67.5
20       0.95     63.4
21       0.99     62.8
22       0.97     62.2
23       1.04     61.2
24       1.11     58.7
25       1.17     56.3
26       1.19     54.2
27       1.13     60.3
28       1.09     57.7
29       1.09     56.4
30       1.07     56.8
31     

## 3. Organic is losing ground ✓

In [18]:
# The organic pricing Premium for 2026 has been change to .0f to .1f to have better accurate percentages
# The Memo will be update to reflect the changes.
header("3. CONVENTIONAL vs. ORGANIC - Total U.S., weeks 1-32")
bytype = summarize(us_ytd, ["year", "Type"])
for t in ["Conventional", "Organic"]:
    a, b = bytype.loc[(2025, t)], bytype.loc[(2026, t)]
    print(f"{t:<13} units {pct(b[VOL], a[VOL]):+.1f}%, dollars "
          f"{pct(b['dollars'], a['dollars']):+.1f}%, price ${a['asp']:.2f} -> ${b['asp']:.2f}")
for y in [2025, 2026]:
    prem = pct(bytype.loc[(y, "Organic"), "asp"], bytype.loc[(y, "Conventional"), "asp"])
    share = bytype.loc[(y, "Organic"), VOL] / tot.loc[y, VOL] * 100
    print(f"{y}: organic price premium {prem:.1f}%, organic share of units {share:.1f}%")


3. CONVENTIONAL vs. ORGANIC - Total U.S., weeks 1-32
Conventional  units +14.5%, dollars -7.8%, price $1.22 -> $0.98
Organic       units -0.7%, dollars -10.9%, price $1.74 -> $1.56
2025: organic price premium 43.2%, organic share of units 5.7%
2026: organic price premium 59.5%, organic share of units 4.9%


## 4. Shoppers are trading up in size

In [19]:
header("4. SIZE (PLU) AND BAG MIX - conventional share of units, weeks 1-32")
mix = conv[conv["Period"] <= YTD_WEEKS].groupby("year")[
    [VOL, "4046", "4225", "4770", "TotalBagged"]].sum()
labels = {"4046": "Small Hass (4046)", "4225": "Large Hass (4225)",
          "4770": "X-Large Hass (4770)", "TotalBagged": "Bagged"}
for col, name in labels.items():
    s = mix[col] / mix[VOL] * 100
    print(f"{name:<20} {s[2025]:.1f}% -> {s[2026]:.1f}%")


4. SIZE (PLU) AND BAG MIX - conventional share of units, weeks 1-32
Small Hass (4046)    48.5% -> 43.5%
Large Hass (4225)    10.6% -> 12.8%
X-Large Hass (4770)  1.4% -> 2.4%
Bagged               39.5% -> 41.3%


## 5. Growth is uneven by region

In [20]:
header("5. REGIONS - all avocados, weeks 1-32")
reg = summarize(df[df["Geography"].isin(REGIONS) & (df["Period"] <= YTD_WEEKS)],
                ["Geography", "year"]).unstack("year")
region_table = pd.DataFrame({
    "unit_growth_%": pct(reg[(VOL, 2026)], reg[(VOL, 2025)]),
    "price_2025": reg[("asp", 2025)],
    "price_2026": reg[("asp", 2026)],
    "dollar_change_%": pct(reg[("dollars", 2026)], reg[("dollars", 2025)]),
}).sort_values("unit_growth_%", ascending=False)
print(region_table.round(2).to_string())
print(f"\nAll regions grew units: {(region_table['unit_growth_%'] > 0).all()}; "
      f"all lost dollars: {(region_table['dollar_change_%'] < 0).all()}")

mkt = df[df["Geography"].isin(SOUTHEAST_MARKETS) & (df["Period"] <= YTD_WEEKS)] \
    .groupby(["Geography", "year"])[VOL].sum().unstack("year")
mkt["added_units_M"] = (mkt[2026] - mkt[2025]) / 1e6
mkt["growth_%"] = pct(mkt[2026], mkt[2025])
print("\nSoutheast markets ranked by added units:")
print(mkt[["added_units_M", "growth_%"]].sort_values("added_units_M", ascending=False)
      .round(1).to_string())


5. REGIONS - all avocados, weeks 1-32
               unit_growth_%  price_2025  price_2026  dollar_change_%
Geography                                                            
Southeast              19.82        1.19        0.90            -9.79
California             14.98        1.43        1.17            -6.00
Midsouth               14.82        1.20        0.99            -4.88
South Central          14.23        1.07        0.84            -9.77
Northeast              13.67        1.19        1.01            -3.73
Great Lakes            12.56        1.20        0.99            -7.34
Plains                 11.32        1.14        0.92           -10.00
West                    6.97        1.40        1.14           -12.95

All regions grew units: True; all lost dollars: True

Southeast markets ranked by added units:
year                   added_units_M  growth_%
Geography                                     
Miami/FT. Lauderdale            12.2      22.7
Atlanta                 

## 6. Outlook

In [21]:
header("6. OUTLOOK - full-year 2026 projection")
wkly = us.groupby(["year", "Period"])[[VOL, "dollars"]].sum()
fy25 = wkly.loc[2025].sum()
rest25 = wkly.loc[2025].loc[YTD_WEEKS + 1:52].sum()            # weeks 33-52 of 2025
ytd26 = wkly.loc[2026].sum()
recent = wkly.loc[2026].loc[29:32].sum() / wkly.loc[2025].loc[29:32].sum()  # last 4 weeks YoY

print(f"FY2025 actual: {fy25[VOL]/1e9:.2f}B units, ${fy25['dollars']/1e9:.2f}B")
print(f"Recent 4-week YoY: units {pct(recent[VOL], 1):+.1f}%, dollars {pct(recent['dollars'], 1):+.1f}%")
print(f"Projection if recent growth continues: "
      f"{(ytd26[VOL] + rest25[VOL] * recent[VOL])/1e9:.2f}B units, "
      f"${(ytd26['dollars'] + rest25['dollars'] * recent['dollars'])/1e9:.2f}B")
print(f"Projection if fall volume just matches 2025: "
      f"{(ytd26[VOL] + rest25[VOL])/1e9:.2f}B units")

thanksgiving = wk.loc[2025].loc[44:52, VOL] / 1e6
print("\n2025 holiday-season conventional volume (millions, weeks 44-52):")
print(thanksgiving.round(1).to_string())


6. OUTLOOK - full-year 2026 projection
FY2025 actual: 2.99B units, $3.49B
Recent 4-week YoY: units +7.0%, dollars -0.2%
Projection if recent growth continues: 3.32B units, $3.31B
Projection if fall volume just matches 2025: 3.24B units

2025 holiday-season conventional volume (millions, weeks 44-52):
Period
44    54.7
45    56.3
46    55.3
47    51.9
48    45.8
49    55.9
50    53.3
51    51.3
52    48.8


## 7. Recommendation support

In [22]:
header("7. ORGANIC VOLUME CHANGE BY REGION - weeks 1-32")
o = df[df["Geography"].isin(REGIONS) & (df["Type"] == "Organic") & (df["Period"] <= YTD_WEEKS)] \
    .groupby(["Geography", "year"])[VOL].sum().unstack("year")
print(pct(o[2026], o[2025]).sort_values().round(1).to_string())


7. ORGANIC VOLUME CHANGE BY REGION - weeks 1-32
Geography
Great Lakes     -17.0
Midsouth        -11.9
West              0.1
Northeast         1.9
Southeast         2.0
South Central     3.1
California        5.9
Plains           20.6
